# LeetCode 386 — Lexicographical Numbers bằng Python

Link đề: https://leetcode.com/problems/lexicographical-numbers/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Trie, Math

Skeleton LeetCode:

```python
class Solution:
    def lexicalOrder(self, n: int) -> List[int]:
        ...
```

Ghi chú: đề yêu cầu thuật toán `O(n)` thời gian và `O(1)` bộ nhớ phụ (không tính mảng kết quả). Mọi lời giải trong notebook đã được đối chiếu với nhau và với một **hàm kiểm tra tính hợp lệ** độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: liệt kê các số `1..n` theo **thứ tự từ điển** (so sánh như **chuỗi**: `"10" < "2"`).
2. Nắm các công cụ Python: so sánh chuỗi, `sorted(..., key=str)`, phép `//` và `%` để "bỏ/đọc chữ số cuối", đệ quy, ngăn xếp bằng `list`, và máy phát `yield`.
3. Đi từ **sắp xếp chuỗi** → **DFS đệ quy trên "cây tiền tố"** → **DFS bằng ngăn xếp** → **nhảy tới số kế tiếp `O(1)` bộ nhớ**.
4. Hiểu mấu chốt: thứ tự từ điển của số chính là **duyệt tiền thứ tự (preorder)** của cây 10 nhánh mà con của `x` là `10x, 10x+1, …, 10x+9`.
5. Tránh các lỗi kinh điển: dùng `<` thay `<=` khi thử xuống tầng sâu hơn, quên điều kiện `cur % 10 == 9`/`cur == n` khi quay lên, bắt đầu DFS từ `0`, và trả về chuỗi thay vì số.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Hãy trả về **tất cả các số trong đoạn `[1, n]` được sắp xếp theo thứ tự từ điển**. Bạn phải viết thuật toán chạy trong `O(n)` thời gian và dùng `O(1)` bộ nhớ phụ.

**Ví dụ 1:** `n = 13` → `[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]`.

**Ví dụ 2:** `n = 2` → `[1, 2]`.

**Ràng buộc:** `1 <= n <= 5 * 10^4`.

"Thứ tự từ điển" nghĩa là so sánh **từng ký tự từ trái sang phải** như các từ trong từ điển: `"1" < "10" < "11" < "2"`, vì `"1"` là tiền tố của `"10"` (ngắn hơn thì đứng trước), còn `"10"` so với `"2"` thì `'1' < '2'`.

In [1]:
from typing import List
import random
import time

n = 13
print("thứ tự số thường :", list(range(1, n + 1)))
print("thứ tự từ điển   :", sorted(range(1, n + 1), key=str))
print("so sánh chuỗi    :", "10" < "2", "1" < "10", "9" < "10", "100" < "11")

thứ tự số thường : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
thứ tự từ điển   : [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
so sánh chuỗi    : True True False True


## 3. Kiến thức nền

### 3.1. So sánh chuỗi (thứ tự từ điển)

Hai chuỗi được so sánh **từng ký tự từ trái sang phải** theo mã ký tự; gặp ký tự khác nhau đầu tiên thì quyết định. Nếu một chuỗi là **tiền tố** của chuỗi kia thì chuỗi ngắn hơn **nhỏ hơn**. Với chữ số, mã tăng dần `'0' < '1' < … < '9'`, nên so sánh chuỗi chữ số cũng "theo từ điển".

In [2]:
for a, b in [("1", "10"), ("10", "2"), ("123", "13"), ("9", "90"), ("100", "1000"), ("19", "2")]:
    print(f"{a!r:>7} < {b!r:<7} ->", a < b)

    '1' < '10'    -> True
   '10' < '2'     -> True
  '123' < '13'    -> True
    '9' < '90'    -> True
  '100' < '1000'  -> True
   '19' < '2'     -> True


### 3.2. `sorted(..., key=...)`

`sorted(seq, key=f)` sắp xếp theo giá trị `f(x)` mà **trả lại chính các phần tử gốc** (số vẫn là số). Với `key=str`, mỗi số được đem so như chuỗi.

In [3]:
nums = [5, 12, 100, 3, 21]
print(sorted(nums))                    # [3, 5, 12, 21, 100]  theo giá trị
print(sorted(nums, key=str))           # [100, 12, 21, 3, 5]  theo từ điển, vẫn là số
print(sorted(map(str, nums)))          # ['100', '12', '21', '3', '5']  -> là CHUỖI (cẩn thận!)

[3, 5, 12, 21, 100]
[100, 12, 21, 3, 5]
['100', '12', '21', '3', '5']


### 3.3. `//` và `%` để thao tác chữ số cuối

- `x * 10`: **thêm** chữ số `0` ở cuối (`12 → 120`).
- `x // 10`: **bỏ** chữ số cuối (`123 → 12`).
- `x % 10`: **chữ số cuối** (`123 → 3`).
- `x + 1`: số "anh em kế tiếp" (cùng tiền tố, chữ số cuối tăng 1) khi `x % 10 != 9`.

In [4]:
x = 123
print(x * 10, x // 10, x % 10, x + 1)       # 1230 12 3 124
print(19 % 10 == 9, 19 + 1)                 # True 20: tăng 19 thành 20 sẽ đổi cả chữ số hàng chục!
print(0 // 10, 9 // 10, 0 % 10)             # 0 0 0

1230 12 3 124
True 20
0 0 0


### 3.4. Cây tiền tố (trie) mười nhánh

Xem mỗi số là một **nút**. Cha của `12` là `1`; con của `x` là `10x, 10x+1, …, 10x+9` (nếu `≤ n`). Các nút ở tầng đầu là `1..9`. Đây là cây trong đó **đường đi từ gốc** tới một nút chính là các chữ số của số đó (một trie của các chuỗi chữ số).

In [5]:
def children(x, n):
    return [10 * x + d for d in range(10) if 10 * x + d <= n]

n = 25
print("tầng 1:", list(range(1, 10)))
for x in (1, 2, 3):
    print(f"con của {x}:", children(x, n))
print("con của 12:", children(12, n), "(rỗng: 120 > 25)")

tầng 1: [1, 2, 3, 4, 5, 6, 7, 8, 9]
con của 1: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
con của 2: [20, 21, 22, 23, 24, 25]
con của 3: []
con của 12: [] (rỗng: 120 > 25)


### 3.5. DFS tiền thứ tự (preorder) và ngăn xếp

DFS tiền thứ tự: **thăm nút trước**, rồi lần lượt các con từ nhỏ tới lớn. Cài bằng đệ quy hoặc bằng **ngăn xếp** (nhớ: ngăn xếp lấy ra **phần tử đẩy vào sau cùng**, nên muốn thăm con nhỏ trước phải **đẩy con lớn trước**).

In [6]:
# DFS preorder thu nhỏ trên cây: 1 -> (10, 11), 2
tree = {1: [10, 11], 10: [], 11: [], 2: []}
order = []
def dfs(u):
    order.append(u)
    for v in tree[u]:
        dfs(v)
for root in (1, 2):
    dfs(root)
print(order)       # [1, 10, 11, 2]

order, stack = [], [2, 1]            # đẩy 2 rồi 1 -> lấy ra 1 trước
while stack:
    u = stack.pop()
    order.append(u)
    stack.extend(reversed(tree[u]))  # con lớn đẩy trước, con nhỏ đẩy sau
print(order)       # [1, 10, 11, 2]

[1, 10, 11, 2]
[1, 10, 11, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tưởng tượng một **cuốn từ điển** mà "từ" là các số. Bạn tra như sau: bắt đầu từ chữ số `1`. Trước khi qua `2`, bạn phải liệt kê **hết mọi số bắt đầu bằng `1`**, và trong nhóm đó lại phải xử lý `10` cùng các số bắt đầu bằng `10`, rồi mới tới `11`...

Tính tay với `n = 13`:

1. Thăm `1`. Con của `1` là `10..19`, nhưng `n = 13` nên chỉ `10, 11, 12, 13` tồn tại.
2. Thăm `10` (con của `10` là `100+` > 13, không có) → `11` → `12` → `13`.
3. Hết con của `1`, quay lên mức đầu: thăm `2` (con `20+` > 13, không có), rồi `3, 4, …, 9`.

Kết quả: `1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9` — đúng ví dụ. Thứ tự này **chính là DFS preorder của cây** mục 3.4.

## 5. Cách 1 — Brute Force / baseline: sắp xếp theo chuỗi

**Ý tưởng:** tạo `1..n`, sắp xếp với `key=str`. Python so sánh chuỗi theo từ điển, và `sorted` giữ nguyên phần tử số.

Một dòng, đúng. Nhưng thời gian `O(n log n)` (nhân thêm chi phí so sánh chuỗi ngắn) và bộ nhớ `O(n)` cho các chuỗi tạm — chưa đạt `O(n)`/`O(1)`.

In [7]:
class SolutionSort:
    """Cách 1: sắp xếp theo chuỗi."""

    def lexicalOrder(self, n: int) -> List[int]:
        return sorted(range(1, n + 1), key=str)


print(SolutionSort().lexicalOrder(13))
print(SolutionSort().lexicalOrder(2))
assert SolutionSort().lexicalOrder(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
assert SolutionSort().lexicalOrder(2) == [1, 2]

[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2]


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ phụ (`key` tạo chuỗi cho mỗi số).

**Nút thắt:** ta **không tận dụng** việc các số đã có cấu trúc "đoán được" (cây tiền tố). Biết cấu trúc đó, ta **sinh trực tiếp** theo đúng thứ tự thay vì sắp xếp.

## 6. Ý tưởng thuật toán / bước đột phá

Thứ tự từ điển của `1..n` = **duyệt preorder** cây trong đó con của `x` là `10x … 10x+9` (nếu `≤ n`) và các nút tầng đầu là `1 … 9`.

Từ đó có hai cách sinh trực tiếp, `O(n)`:

- **DFS đệ quy/ngăn xếp:** thăm `x`, rồi thăm các con theo thứ tự tăng. Mỗi số được thăm **đúng một lần**.
- **Nhảy tới số kế tiếp (không ngăn xếp):** từ số `cur` đang đứng, số kế tiếp trong thứ tự preorder là:
  1. Nếu `cur * 10 <= n` ⇒ **đi xuống con đầu tiên**: `cur = cur * 10`.
  2. Ngược lại phải sang **anh em kế**. Nhưng nếu `cur` đã là con **cuối** của cha (`cur % 10 == 9`) hoặc `cur + 1 > n` (tức `cur == n`) thì **quay lên cha** (`cur //= 10`) và thử lại.
  3. Khi đã tìm được `cur` có thể tăng, làm `cur += 1`.

Mỗi lần **quay lên** ta bỏ một chữ số, và mỗi nút chỉ bị "rời" một lần ⇒ tổng số bước **amortized `O(n)`**, bộ nhớ phụ `O(1)` (chỉ biến `cur`).

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (sắp xếp chuỗi, mục 5), Cách 2 (DFS đệ quy), Cách 3 (nhảy số kế tiếp `O(1)` bộ nhớ — **chính thức**), Cách 4 (DFS bằng ngăn xếp tường minh).

### Cách 2 — DFS đệ quy trên cây tiền tố

Giải thích từng đoạn:

1. Với mỗi chữ số đầu `d = 1..9` gọi `dfs(d)`.
2. `dfs(x)`: nếu `x > n` thì dừng (không tồn tại). Ngược lại `ans.append(x)` (thăm trước), rồi với `d = 0..9` gọi `dfs(10*x + d)`.

Có thể dừng sớm: nếu `10*x + d > n` thì mọi `d` lớn hơn cũng vượt `n` nên `break`.

In [8]:
class SolutionDFS:
    """Cách 2: DFS đệ quy (preorder) trên cây 10 nhánh."""

    def lexicalOrder(self, n: int) -> List[int]:
        ans = []

        def dfs(x: int) -> None:
            if x > n:
                return
            ans.append(x)                 # thăm nút trước
            for d in range(10):           # rồi các con theo thứ tự tăng
                if 10 * x + d > n:
                    break                 # các con sau còn lớn hơn
                dfs(10 * x + d)

        for first in range(1, 10):
            dfs(first)
        return ans


print(SolutionDFS().lexicalOrder(13))
assert SolutionDFS().lexicalOrder(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
assert SolutionDFS().lexicalOrder(2) == [1, 2]

[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]


**Code nộp LeetCode cho Cách 2:**

```python
class Solution:
    def lexicalOrder(self, n: int) -> List[int]:
        ans = []

        def dfs(x: int) -> None:
            if x > n:
                return
            ans.append(x)
            for d in range(10):
                if 10 * x + d > n:
                    break
                dfs(10 * x + d)

        for first in range(1, 10):
            dfs(first)
        return ans
```

**Độ phức tạp:** `O(n)` thời gian (mỗi số thăm một lần), bộ nhớ phụ `O(log n)` cho ngăn xếp đệ quy (độ sâu = số chữ số, tối đa 5 ở `n ≤ 5·10^4`) — vẫn không phải `O(1)` thuần.

---

### Cách 3 — Nhảy tới số kế tiếp, `O(1)` bộ nhớ — chính thức

Giải thích từng đoạn (khởi đầu `cur = 1`, lặp đúng `n` lần):

1. `ans.append(cur)` — ghi số hiện tại.
2. Nếu `cur * 10 <= n`: **xuống con đầu**: `cur *= 10`.
3. Ngược lại: **quay lên** trong lúc `cur % 10 == 9` (đã là con cuối, không có anh em kế) **hoặc** `cur == n` (anh em kế sẽ vượt `n`): `cur //= 10`.
4. Sau đó `cur += 1` (sang anh em kế của nút vừa dừng).

Hai điều kiện của `while` là then chốt, ta sẽ thấy lỗi nếu thiếu ở mục 13.

In [9]:
class SolutionIter:
    """Cách 3 (demo): nhảy tới số kế tiếp trong thứ tự từ điển, O(1) bộ nhớ phụ."""

    def lexicalOrder(self, n: int) -> List[int]:
        ans = []
        cur = 1
        for _ in range(n):
            ans.append(cur)
            if cur * 10 <= n:                       # có con -> xuống con đầu
                cur *= 10
            else:
                while cur % 10 == 9 or cur == n:    # hết anh em -> quay lên
                    cur //= 10
                cur += 1                            # sang anh em kế
        return ans


print(SolutionIter().lexicalOrder(13))
assert SolutionIter().lexicalOrder(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
assert SolutionIter().lexicalOrder(2) == [1, 2]
assert SolutionIter().lexicalOrder(1) == [1]

[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]


**Code nộp LeetCode cho Cách 3:**

```python
class Solution:
    def lexicalOrder(self, n: int) -> List[int]:
        ans = []
        cur = 1
        for _ in range(n):
            ans.append(cur)
            if cur * 10 <= n:
                cur *= 10
            else:
                while cur % 10 == 9 or cur == n:
                    cur //= 10
                cur += 1
        return ans
```

**Độ phức tạp:** `O(n)` thời gian (amortized, mỗi bước quay lên "trả" bằng một bước xuống trước đó), bộ nhớ phụ `O(1)`; chỉ mảng kết quả `O(n)`.

---

### Cách 4 — DFS bằng ngăn xếp tường minh

**Ý tưởng:** cùng thứ tự preorder nhưng không đệ quy. Đẩy `9, 8, …, 1` vào ngăn xếp (để `1` ở đỉnh). Lặp: lấy `x` ra, ghi `x`, rồi đẩy các con của `x` theo thứ tự **giảm dần** (`10x+9` trước … `10x` sau) để con nhỏ nhất được lấy ra trước.

Bộ nhớ ngăn xếp tối đa ~`9 × độ_sâu` ≈ `45` phần tử với `n ≤ 5·10^4` (mỗi tầng đẩy tối đa 10 con) — tức `O(log n)` nhưng với hằng số nhỏ, và **không có đệ quy**.

In [10]:
class SolutionStack:
    """Cách 4: DFS preorder bằng ngăn xếp."""

    def lexicalOrder(self, n: int) -> List[int]:
        ans = []
        stack = [d for d in range(min(9, n), 0, -1)]   # đỉnh ngăn xếp là 1
        while stack:
            x = stack.pop()
            ans.append(x)
            for d in range(9, -1, -1):                 # đẩy con lớn trước, nhỏ sau
                child = 10 * x + d
                if child <= n:
                    stack.append(child)
        return ans


print(SolutionStack().lexicalOrder(13))
assert SolutionStack().lexicalOrder(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
assert SolutionStack().lexicalOrder(2) == [1, 2]

[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]


**Code nộp LeetCode cho Cách 4:**

```python
class Solution:
    def lexicalOrder(self, n: int) -> List[int]:
        ans = []
        stack = [d for d in range(min(9, n), 0, -1)]
        while stack:
            x = stack.pop()
            ans.append(x)
            for d in range(9, -1, -1):
                child = 10 * x + d
                if child <= n:
                    stack.append(child)
        return ans
```

**Độ phức tạp:** `O(n)` thời gian (mỗi số vào/ra ngăn xếp một lần; mỗi lần thăm xét 10 con ⇒ hằng số 10), bộ nhớ phụ `O(log n)` (ngăn xếp).

## 8. TRACE từng bước

Trace Cách 3 với `n = 25` (đủ để thấy cả **xuống**, **sang anh em** và **quay lên**). Mỗi dòng: bước, `cur` được ghi, nhánh rẽ tiếp theo và `cur` mới.

Với `n = 25` chuỗi kết quả là `1, 10, 11, …, 19, 2, 20, 21, …, 25, 3, 4, …, 9`.

| Bước | `cur` ghi | Quyết định | `cur` mới |
|---|---|---|---|
| 1 | 1 | `10 <= 25` ⇒ xuống | 10 |
| 2 | 10 | `100 > 25`, `10 % 10 = 0` ⇒ sang anh em | 11 |
| 3..10 | 11..18 | sang anh em (+1) | … |
| 11 | 19 | `190 > 25`, `19 % 10 == 9` ⇒ quay lên `1`; `1 % 10 != 9` ⇒ dừng; `+1` | 2 |
| 12 | 2 | `20 <= 25` ⇒ xuống | 20 |
| 13 | 20 | sang anh em | 21 |
| 14..17 | 21..24 | sang anh em | … |
| 18 | 25 | `250 > 25`, `cur == n` ⇒ quay lên `2`; `+1` | 3 |
| 19 | 3 | `30 > 25`, `3 % 10 != 9` ⇒ sang anh em | 4 |

Hàm dưới in từng bước thật.

In [11]:
def trace_iter(n, max_rows=None):
    print(f"n = {n}")
    print(f"{'bước':<6}{'cur':<6}{'quyết định':<46}{'cur mới'}")
    cur, out = 1, []
    for step in range(1, n + 1):
        out.append(cur)
        if cur * 10 <= n:
            why = f"{cur}*10={cur * 10} <= {n}: xuống con đầu"
            cur *= 10
        else:
            ups = []
            while cur % 10 == 9 or cur == n:
                why_up = f"{cur}%10==9" if cur % 10 == 9 else f"{cur}==n"
                ups.append(why_up)
                cur //= 10
            why = ("quay lên (" + ", ".join(ups) + ") rồi +1") if ups else "sang anh em (+1)"
            cur += 1
        if max_rows is None or step <= max_rows or step > n - 3:
            print(f"{step:<6}{out[-1]:<6}{why:<46}{cur}")
        elif step == max_rows + 1:
            print("...")
    print("=>", out, "\n")
    return out


out = trace_iter(25)
assert out == SolutionSort().lexicalOrder(25)

n = 25
bước  cur   quyết định                                    cur mới
1     1     1*10=10 <= 25: xuống con đầu                  10
2     10    sang anh em (+1)                              11
3     11    sang anh em (+1)                              12
4     12    sang anh em (+1)                              13
5     13    sang anh em (+1)                              14
6     14    sang anh em (+1)                              15
7     15    sang anh em (+1)                              16
8     16    sang anh em (+1)                              17
9     17    sang anh em (+1)                              18
10    18    sang anh em (+1)                              19
11    19    quay lên (19%10==9) rồi +1                    2
12    2     2*10=20 <= 25: xuống con đầu                  20
13    20    sang anh em (+1)                              21
14    21    sang anh em (+1)                              22
15    22    sang anh em (+1)                              23
16    23    s

Với `n = 13` (ví dụ đề), ta xem nhánh **quay lên** hai lần khi gặp `9`:

In [12]:
out = trace_iter(13)
assert out == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]

n = 13
bước  cur   quyết định                                    cur mới
1     1     1*10=10 <= 13: xuống con đầu                  10
2     10    sang anh em (+1)                              11
3     11    sang anh em (+1)                              12
4     12    sang anh em (+1)                              13
5     13    quay lên (13==n) rồi +1                       2
6     2     sang anh em (+1)                              3
7     3     sang anh em (+1)                              4
8     4     sang anh em (+1)                              5
9     5     sang anh em (+1)                              6
10    6     sang anh em (+1)                              7
11    7     sang anh em (+1)                              8
12    8     sang anh em (+1)                              9
13    9     quay lên (9%10==9) rồi +1                     1
=> [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9] 



Và một trường hợp quay lên **nhiều tầng** liên tiếp: `n = 199` (sau `199` quay lên `19` rồi `1` rồi mới sang `2`). Chỉ in vài dòng cuối:

In [13]:
cur, n = 199, 199
ups = 0
while cur % 10 == 9 or cur == n:
    print(f"quay lên: cur = {cur}  ->  {cur // 10}")
    cur //= 10
    ups += 1
cur += 1
print("sang anh em: cur =", cur, f"(quay lên {ups} tầng)")
assert cur == 2

quay lên: cur = 199  ->  19
quay lên: cur = 19  ->  1
sang anh em: cur = 2 (quay lên 2 tầng)


## 9. Code nộp LeetCode

Phiên bản cuối (Cách 3 — `O(1)` bộ nhớ phụ, đúng yêu cầu đề), đúng tên lớp/method/signature:

In [14]:
class Solution:
    def lexicalOrder(self, n: int) -> List[int]:
        ans = []
        cur = 1
        for _ in range(n):
            ans.append(cur)
            if cur * 10 <= n:
                cur *= 10
            else:
                while cur % 10 == 9 or cur == n:
                    cur //= 10
                cur += 1
        return ans


print(Solution().lexicalOrder(13))
print(Solution().lexicalOrder(2))

[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2]


## 10. Vì sao thuật toán đúng

Thứ tự từ điển của `1..n` **bằng** thứ tự **preorder** của cây mười nhánh (cha của `x` là `x // 10`, các con theo thứ tự `10x, …, 10x+9`): vì với hai số khác nhau, hoặc số này là **tiền tố** của số kia (cha/tổ tiên đứng trước — đúng preorder), hoặc chúng khác nhau tại chữ số đầu tiên (nhánh có chữ số nhỏ hơn đứng trước — đúng thứ tự thăm con).

Cách 3 mô phỏng "nút kế tiếp trong preorder":

1. Nếu nút có con (`cur·10 ≤ n`): nút kế là con **đầu tiên** `cur·10`.
2. Nếu không có con: nút kế là **anh em kế** `cur+1` **nếu** nó tồn tại và vẫn cùng cha — tức `cur % 10 != 9` (nếu `cur` tận cùng 9 thì `cur+1` đổi chữ số hàng chục, không cùng cha) và `cur + 1 ≤ n` (tức `cur != n`).
3. Nếu không có anh em kế: **quay lên cha** (`cur // 10`) và lặp lại bước 2 cho cha (cha đã được thăm rồi, ta chỉ tìm anh em kế của cha). Nếu quay lên tới `0` thì đã thăm hết cây (xảy ra đúng ở bước thứ `n`, sau khi ghi số cuối).

Vì mỗi số trong `1..n` là **một nút duy nhất** và preorder thăm **mỗi nút đúng một lần**, vòng lặp `n` lần ghi đúng `n` số khác nhau theo đúng thứ tự.

**Độ phức tạp:** mỗi lần quay lên bỏ một chữ số mà trước đó đã có một bước "xuống" thêm chữ số đó; tổng số bước xuống `≤ n` ⇒ tổng số bước quay lên `≤ n`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp theo chuỗi | `O(n log n)` | `O(n)` | Đơn giản nhất; chưa đạt yêu cầu đề |
| 2. DFS đệ quy | `O(n)` | `O(log n)` đệ quy | Độ sâu = số chữ số ≤ 5 |
| 3. Nhảy số kế tiếp | `O(n)` amortized | **`O(1)`** | Đúng yêu cầu đề |
| 4. DFS ngăn xếp | `O(n)` | `O(log n)` | Không đệ quy |

("Space phụ" không tính mảng kết quả `O(n)` mà đề bắt buộc trả về.)

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** `n = 13`, `n = 2`.
2. **Edge case:** `n = 1`, `n = 9`, `n = 10`, `n = 19`, `n = 20`, `n = 99`, `n = 100`, `n = 101`, `n = 999`, `n = 1000`, `n = 50000` (biên tối đa).
3. **Hàm kiểm tra tính hợp lệ độc lập:** `is_valid_lex(res, n)` — là hoán vị của `1..n` và **mỗi cặp liền kề** thỏa `str(a) < str(b)`.
4. **Đối chiếu 4 cách bằng `assert`:** vét cạn **mọi `n` từ 1 đến 3000**, cộng 200 giá trị ngẫu nhiên (seed cố định) trong `[1, 50000]`.
5. **Oracle thứ hai:** sắp xếp bằng `cmp_to_key` so sánh chuỗi (độc lập với `key=str`).

In [15]:
from functools import cmp_to_key

def is_valid_lex(res, n):
    """Kiểm tra độc lập: hoán vị của 1..n và mỗi cặp liền kề tăng theo thứ tự chuỗi."""
    if len(res) != n or sorted(res) != list(range(1, n + 1)):
        return False
    return all(str(a) < str(b) for a, b in zip(res, res[1:]))


def oracle_cmp(n):
    def cmp(a, b):
        sa, sb = str(a), str(b)
        return -1 if sa < sb else (1 if sa > sb else 0)
    return sorted(range(1, n + 1), key=cmp_to_key(cmp))


IMPLS = [SolutionSort, SolutionDFS, SolutionIter, SolutionStack, Solution]

# 1) Ví dụ đề
for cls in IMPLS:
    assert cls().lexicalOrder(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
    assert cls().lexicalOrder(2) == [1, 2]
print("Ví dụ đề: OK")

# 2) Edge cases + hàm kiểm tra hợp lệ + oracle thứ hai
for n in [1, 9, 10, 19, 20, 99, 100, 101, 109, 110, 999, 1000, 1001, 9999, 10000, 50000]:
    expected = oracle_cmp(n)
    for cls in IMPLS:
        res = cls().lexicalOrder(n)
        assert is_valid_lex(res, n), (cls.__name__, n)
        assert res == expected, (cls.__name__, n)
print("Edge cases (kể cả n = 50000): OK")

Ví dụ đề: OK


Edge cases (kể cả n = 50000): OK


In [16]:
# 3) Vét cạn mọi n từ 1 đến 3000: mọi cách phải bằng nhau và hợp lệ
for n in range(1, 3001):
    base = SolutionSort().lexicalOrder(n)
    assert is_valid_lex(base, n)
    for cls in IMPLS[1:]:
        assert cls().lexicalOrder(n) == base, (cls.__name__, n)
print("Vét cạn n = 1..3000, 5 cách: OK")

# 4) Ngẫu nhiên (seed cố định) trong khoảng ràng buộc
rng = random.Random(386)
for _ in range(200):
    n = rng.randint(1, 50000)
    base = SolutionSort().lexicalOrder(n)
    for cls in IMPLS[1:]:
        assert cls().lexicalOrder(n) == base, (cls.__name__, n)
print("200 giá trị n ngẫu nhiên trong [1, 50000]: OK")

Vét cạn n = 1..3000, 5 cách: OK


200 giá trị n ngẫu nhiên trong [1, 50000]: OK


In [17]:
# 5) Tính chất thêm: phần tử đầu luôn là 1; phần tử cuối có chuỗi lớn nhất
rng = random.Random(1386)
for _ in range(200):
    n = rng.randint(1, 50000)
    res = Solution().lexicalOrder(n)
    assert res[0] == 1
    assert all(type(x) is int for x in res)            # là SỐ, không phải chuỗi
    assert max(res) == n and min(res) == 1
    # số cuối cùng trong thứ tự từ điển phải là số có chuỗi lớn nhất
    assert str(res[-1]) == max(map(str, res)), (n, res[-1])
print("Tính chất: OK")

# 6) Hiệu năng (chỉ in): n = 50000
n = 50000
for cls in IMPLS:
    t = time.perf_counter()
    for _ in range(20):
        cls().lexicalOrder(n)
    print(f"{cls.__name__:<14} {(time.perf_counter() - t) / 20 * 1000:.2f} ms / lần")

Tính chất: OK
SolutionSort   3.64 ms / lần


SolutionDFS    10.08 ms / lần
SolutionIter   5.09 ms / lần


SolutionStack  23.06 ms / lần
Solution       4.22 ms / lần


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả về **chuỗi** thay vì số

`sorted(map(str, range(1, n+1)))` cho `['1', '10', ...]` — LeetCode so sánh với số nguyên nên **sai kiểu**.

In [18]:
n = 13
wrong = sorted(map(str, range(1, n + 1)))
print(wrong)
print(wrong == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9], "(chuỗi != số)")
right = sorted(range(1, n + 1), key=str)          # giữ nguyên số, chỉ so sánh theo chuỗi
print(right)

['1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']
False (chuỗi != số)
[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]


**Sửa:** dùng `key=str`, hoặc `list(map(int, sorted(map(str, ...))))`.

### Lỗi 2: dùng `<` thay vì `<=` khi thử xuống con đầu

`if cur * 10 < n` bỏ sót trường hợp `cur * 10 == n` (ví dụ `n = 10`: nút `10` là con đầu của `1`).

In [19]:
def bad_strict(n):
    ans, cur = [], 1
    for _ in range(n):
        ans.append(cur)
        if cur * 10 < n:                       # SAI: phải là <=
            cur *= 10
        else:
            while cur % 10 == 9 or cur == n:
                cur //= 10
            cur += 1
    return ans

print(bad_strict(10), "(đúng: [1, 10, 2, 3, 4, 5, 6, 7, 8, 9])")
print(Solution().lexicalOrder(10))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 1] (đúng: [1, 10, 2, 3, 4, 5, 6, 7, 8, 9])
[1, 10, 2, 3, 4, 5, 6, 7, 8, 9]


### Lỗi 3: quên điều kiện `cur == n` khi quay lên

Khi `cur == n`, `cur + 1` vượt `n` — nhưng nếu chỉ kiểm tra `cur % 10 == 9` thì vẫn `+1` và sinh ra số **lớn hơn `n`**.

In [20]:
def bad_no_n_check(n):
    ans, cur = [], 1
    for _ in range(n):
        ans.append(cur)
        if cur * 10 <= n:
            cur *= 10
        else:
            while cur % 10 == 9:              # SAI: thiếu 'or cur == n'
                cur //= 10
            cur += 1
    return ans

res = bad_no_n_check(13)
print(res, "(có số > 13)")
print([x for x in res if x > 13])

[1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 3] (có số > 13)
[14, 15, 16, 17, 18, 19]


### Lỗi 4: quên điều kiện `cur % 10 == 9`

Khi `cur = 19` mà chỉ `+1` ta nhảy sang `20` — số **không phải anh em** của `19` (khác cha). Thứ tự đúng sau `19` (với `n = 25`) là `2`.

In [21]:
def bad_no_nine_check(n):
    ans, cur = [], 1
    for _ in range(n):
        ans.append(cur)
        if cur * 10 <= n:
            cur *= 10
        else:
            while cur == n:                    # SAI: thiếu 'cur % 10 == 9 or'
                cur //= 10
            cur += 1
    return ans

print(bad_no_nine_check(25)[:14])
print(Solution().lexicalOrder(25)[:14], "(đúng)")

[1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
[1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21] (đúng)


### Lỗi 5: DFS bắt đầu từ `0`

Gọi `dfs(0)` hoặc lặp `first` từ `0` ⇒ ghi cả `0` (không thuộc `[1, n]`), và con của `0` là `0..9` lặp lại các chữ số đầu — số bị thừa/trùng.

In [22]:
def bad_start_zero(n):
    ans = []
    def dfs(x):
        if x > n:
            return
        ans.append(x)
        for d in range(10):
            child = 10 * x + d
            if x > 0 and child <= n:          # (với x = 0 phải chặn, nếu không lặp vô hạn 0 -> 0)
                dfs(child)
    for first in range(0, 10):                 # SAI: bắt đầu từ 0
        dfs(first)
    return ans

print(bad_start_zero(13), "(có số 0 không thuộc [1, n])")

[0, 1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9] (có số 0 không thuộc [1, n])


**Sửa:** mức đầu chỉ `1..9`; con của `x` là `10x + d` với `d = 0..9`, và **dừng ngay** khi `10x + d > n`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Thứ tự từ điển của số"** ⇒ nghĩ **cây tiền tố mười nhánh** + **DFS preorder**.
- Muốn `O(1)` bộ nhớ ⇒ mô phỏng DFS bằng **quy tắc "nút kế tiếp"**: xuống (`*10`) → sang anh em (`+1`) → quay lên (`//10`) khi hết anh em.
- Hai điều kiện quay lên **phải thuộc lòng**: `cur % 10 == 9` (hết anh em cùng cha) **hoặc** `cur == n` (anh em kế vượt `n`).
- Kiểm tra bằng **hàm hợp lệ độc lập**: hoán vị của `1..n` + mọi cặp liền kề `str(a) < str(b)`.
- Khi nghi ngờ, **vét cạn** mọi `n` nhỏ (1..3000) so với `sorted(key=str)`.
- Họ bài này: K-th Smallest in Lexicographical Order (440), Lexicographically Smallest Equivalent..., các bài trie/DFS.

## 15. Biến thể của bài

1. **LeetCode 440 — K-th Smallest in Lexicographical Order:** `n` lên tới `10^9`, cần số thứ `k` mà **không** liệt kê hết: đếm kích thước **cây con** của tiền tố để nhảy.
2. **Sinh lười (generator) với `n` rất lớn:** dùng `yield` để không giữ cả mảng.
3. **Khóa số học thay chuỗi:** gán mỗi số `x` khóa `x * 10^(D - len(x))` (đệm `0` cho đủ `D` chữ số) rồi sắp xếp (hòa thì số nhỏ trước) — tránh tạo chuỗi.
4. **Thứ tự từ điển ngược** (giảm dần): duyệt preorder nhưng thăm con theo thứ tự giảm.

Cài thử các biến thể 1, 2, 3:

In [23]:
def subtree_size(prefix, n):
    """Số nút trong cây con gốc prefix (prefix và mọi số bắt đầu bằng prefix, <= n)."""
    cnt, lo, hi = 0, prefix, prefix
    while lo <= n:
        cnt += min(hi, n) - lo + 1
        lo, hi = lo * 10, hi * 10 + 9
    return cnt


def kth_lex(n, k):
    """LeetCode 440: số thứ k (đếm từ 1) trong thứ tự từ điển của 1..n, không liệt kê hết."""
    cur = 1
    k -= 1                                   # đã đứng ở số thứ 1
    while k > 0:
        size = subtree_size(cur, n)
        if size <= k:                        # bỏ cả cây con của cur, sang anh em
            k -= size
            cur += 1
        else:                                # k nằm trong cây con: xuống con đầu
            k -= 1
            cur *= 10
    return cur


for n in (13, 25, 100, 199, 1000):
    full = SolutionSort().lexicalOrder(n)
    for k in range(1, n + 1):
        assert kth_lex(n, k) == full[k - 1], (n, k)
assert kth_lex(10**9, 10**9) == 999999999
print("kth_lex OK; n=13, k=2 ->", kth_lex(13, 2))


def lexical_gen(n):
    cur = 1
    for _ in range(n):
        yield cur
        if cur * 10 <= n:
            cur *= 10
        else:
            while cur % 10 == 9 or cur == n:
                cur //= 10
            cur += 1


gen = lexical_gen(10**9)                     # n khổng lồ: chỉ lấy 12 số đầu
print([next(gen) for _ in range(12)])


def lexical_padded_key(n):
    D = len(str(n))
    return sorted(range(1, n + 1), key=lambda x: (x * 10 ** (D - len(str(x))), x))


for n in (13, 99, 100, 1234):
    assert lexical_padded_key(n) == SolutionSort().lexicalOrder(n), n
print("khóa số học (đệm 0) OK")

kth_lex OK; n=13, k=2 -> 10
[1, 10, 100, 1000, 10000, 100000, 1000000, 10000000, 100000000, 1000000000, 100000001, 100000002]
khóa số học (đệm 0) OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp chuỗi | `sorted(key=str)` | `O(n log n)` | `O(n)` | Viết nhanh khi `n` nhỏ; làm oracle |
| 2. DFS đệ quy | preorder trên cây 10 nhánh | `O(n)` | `O(log n)` | Dễ hiểu nhất, độ sâu rất nhỏ |
| 3. Nhảy số kế tiếp | xuống / sang anh em / quay lên | `O(n)` | **`O(1)`** | **Đúng yêu cầu đề**, phỏng vấn |
| 4. DFS ngăn xếp | preorder bằng stack, đẩy con lớn trước | `O(n)` | `O(log n)` | Không muốn đệ quy, chưa nhớ quy tắc nhảy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code. (a) `"9" < "10"`? (b) `sorted([5, 12, 100, 3], key=str)`; (c) `foo = 1999; foo % 10 == 9`? (d) `Solution().lexicalOrder(1)`.

**Bài 2 (tính tay).** Với `n = 21`, viết thứ tự từ điển đầy đủ rồi kể ra **từng lần quay lên** của Cách 3 (ở `cur` nào, vì sao).

**Bài 3 (điền TODO).** Hoàn thành `lex_todo(n)` theo Cách 3 (`O(1)` bộ nhớ phụ).

In [24]:
def lex_todo(n):
    # TODO 1: ans = [], cur = 1
    # TODO 2: lặp n lần: ghi cur vào ans
    # TODO 3: nếu cur * 10 <= n thì cur *= 10
    # TODO 4: ngược lại, trong khi cur % 10 == 9 hoặc cur == n thì cur //= 10; rồi cur += 1
    return None

# KHÔNG gọi lex_todo ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `class SolutionDFS2` theo Cách 2 (DFS đệ quy) **mà không nhìn lại** mục 7; kiểm tra với `n` từ `1` đến `500` bằng `is_valid_lex`.

**Bài 5 (biến thể / mini challenge).** Viết `lexical_desc(n)` trả `1..n` theo **thứ tự từ điển giảm dần** (`n = 13` → `[9, 8, 7, 6, 5, 4, 3, 2, 13, 12, 11, 10, 1]`). Gợi ý: preorder nhưng thăm **con lớn trước và nút gốc SAU các con** (đó là postorder ngược — hãy tự suy ra!). Kiểm tra bằng cách so với `sorted(range(1, n+1), key=str, reverse=True)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `False` (`'9' > '1'`); (b) `[100, 12, 3, 5]`; (c) `True`; (d) `[1]`. Bài 3: mở đầu `cur = 1`, vòng `for _ in range(n)`.
- **Mức 2 (vừa):** Bài 2: thứ tự `1,10,11,…,19,2,20,21,3,4,…,9`; quay lên tại `19` (`19 % 10 == 9`, về `1`, `+1 → 2`) và tại `21` (`21 == n`, về `2`, `+1 → 3`). Bài 4: `for first in range(1, 10)`; `if 10*x + d > n: break`.
- **Mức 3 (sâu):** Bài 5: thứ tự giảm dần từ điển = đảo ngược dãy tăng dần ⇒ chỉ cần `lexical_desc(n) == lexicalOrder(n)[::-1]`; có thể tự sinh bằng **postorder** của cây với con thăm từ lớn đến nhỏ (đã thăm hết cây con rồi mới ghi nút).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert ("9" < "10") is False
assert sorted([5, 12, 100, 3], key=str) == [100, 12, 3, 5]
assert 1999 % 10 == 9
assert Solution().lexicalOrder(1) == [1]

# Bài 2
assert Solution().lexicalOrder(21) == [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 3, 4, 5, 6, 7, 8, 9]


# Bài 3
def lex_todo_ans(n):
    ans, cur = [], 1
    for _ in range(n):
        ans.append(cur)
        if cur * 10 <= n:
            cur *= 10
        else:
            while cur % 10 == 9 or cur == n:
                cur //= 10
            cur += 1
    return ans


for n in range(1, 400):
    assert lex_todo_ans(n) == SolutionSort().lexicalOrder(n)


# Bài 4
class SolutionDFS2:
    def lexicalOrder(self, n):
        ans = []

        def dfs(x):
            if x > n:
                return
            ans.append(x)
            for d in range(10):
                if 10 * x + d > n:
                    break
                dfs(10 * x + d)

        for first in range(1, 10):
            dfs(first)
        return ans


for n in range(1, 501):
    assert is_valid_lex(SolutionDFS2().lexicalOrder(n), n)


# Bài 5
def lexical_desc(n):
    ans = []

    def post(x):                              # postorder, con thăm từ lớn đến nhỏ
        if x > n:
            return
        for d in range(9, -1, -1):
            if 10 * x + d <= n:
                post(10 * x + d)
        ans.append(x)

    for first in range(9, 0, -1):
        post(first)
    return ans


assert lexical_desc(13) == [9, 8, 7, 6, 5, 4, 3, 2, 13, 12, 11, 10, 1]
for n in range(1, 600):
    assert lexical_desc(n) == sorted(range(1, n + 1), key=str, reverse=True)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nhảy số kế tiếp trong preorder của cây tiền tố)

```python
class Solution:
    def lexicalOrder(self, n: int) -> List[int]:
        ans = []
        cur = 1
        for _ in range(n):
            ans.append(cur)
            if cur * 10 <= n:                       # xuống con đầu
                cur *= 10
            else:
                while cur % 10 == 9 or cur == n:    # hết anh em -> quay lên
                    cur //= 10
                cur += 1                            # sang anh em kế
        return ans
```

**Checklist**

- [ ] Điều kiện xuống là **`cur * 10 <= n`** (có dấu `=`)?
- [ ] Khi quay lên dùng **cả hai** điều kiện: `cur % 10 == 9` **và** `cur == n`?
- [ ] Vòng lặp chạy đúng **`n` lần** (mỗi số một lần), kết quả là **số nguyên**, không phải chuỗi?
- [ ] Không bắt đầu DFS từ `0`; mức đầu là `1..9`?
- [ ] Hiểu "thứ tự từ điển = preorder của cây mười nhánh"?
- [ ] Test: `n = 1`, `9`, `10`, `19`, `20`, `100`, `n = 50000`, vét cạn `n ≤ 3000` đối chiếu `sorted(key=str)`?